# Gold Layer - Regional Summary

**Purpose:** Create regional aggregations and identify medical deserts in Ghana

**Aggregations by Region:**
* Facility counts by type (hospitals, clinics, pharmacies, NGOs)
* Capacity metrics (total beds, total doctors)
* Coverage flags (emergency care, maternal care, pediatric care)
* Specialty diversity
* Volunteer acceptance
* Medical desert flagging

**Input:** `virtue_foundation.ghana.facilities_silver`

**Output:** `virtue_foundation.ghana.regional_summary`

**Medical Desert Criteria:**
A region is flagged as a medical desert if ANY of:
* Total facilities < 3
* Zero hospitals AND < 2 clinics
* Zero doctors AND zero bed capacity

## 1. Setup and Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

# Configuration
CATALOG = "virtue_foundation"
SCHEMA = "ghana"
SOURCE_TABLE = "facilities_silver"
TARGET_TABLE = "regional_summary"

print(f"Source: {CATALOG}.{SCHEMA}.{SOURCE_TABLE}")
print(f"Target: {CATALOG}.{SCHEMA}.{TARGET_TABLE}")

## 2. Load Silver Layer Data

In [0]:
# Read Silver table
df_silver = spark.table(f"{CATALOG}.{SCHEMA}.{SOURCE_TABLE}")

print(f"✅ Loaded Silver table")
print(f"Total records: {df_silver.count():,}")
print(f"Unique regions: {df_silver.select('address_stateOrRegion').distinct().count()}")

## 3. Build Regional Aggregations

Aggregate all metrics by region:
* Facility counts
* Capacity metrics
* Service availability
* Specialty diversity

In [0]:
# Build comprehensive regional aggregations
df_regional = df_silver.groupBy("address_stateOrRegion").agg(
    # ========================================
    # FACILITY COUNTS
    # ========================================
    F.count("*").alias("total_facilities"),
    
    F.sum(
        F.when(F.col("facilityTypeId") == "hospital", 1).otherwise(0)
    ).alias("total_hospitals"),
    
    F.sum(
        F.when(F.col("facilityTypeId") == "clinic", 1).otherwise(0)
    ).alias("total_clinics"),
    
    F.sum(
        F.when(F.col("facilityTypeId") == "pharmacy", 1).otherwise(0)
    ).alias("total_pharmacies"),
    
    F.sum(
        F.when(F.col("facilityTypeId") == "dentist", 1).otherwise(0)
    ).alias("total_dentists"),
    
    F.sum(
        F.when(F.col("facilityTypeId") == "doctor", 1).otherwise(0)
    ).alias("total_doctor_offices"),
    
    F.sum(
        F.when(F.col("organization_type") == "ngo", 1).otherwise(0)
    ).alias("total_ngos"),
    
    # ========================================
    # CAPACITY METRICS
    # ========================================
    F.sum(
        F.coalesce(F.col("capacity"), F.lit(0))
    ).alias("total_bed_capacity"),
    
    F.sum(
        F.coalesce(F.col("numberDoctors"), F.lit(0))
    ).alias("total_doctors"),
    
    F.sum(
        F.when((F.col("numberDoctors").isNotNull()) & (F.col("numberDoctors") > 0), 1).otherwise(0)
    ).alias("facilities_with_doctors"),
    
    F.sum(
        F.when((F.col("capacity").isNotNull()) & (F.col("capacity") > 0), 1).otherwise(0)
    ).alias("facilities_with_beds"),
    
    # ========================================
    # DATA QUALITY METRICS
    # ========================================
    F.avg("completeness_score").alias("avg_completeness_score"),
    
    F.sum(
        F.when(F.col("has_any_contact"), 1).otherwise(0)
    ).alias("facilities_with_contact"),
    
    # ========================================
    # SERVICE AVAILABILITY FLAGS
    # ========================================
    F.max(
        F.when(F.col("acceptsVolunteers") == True, 1).otherwise(0)
    ).alias("facilities_accepting_volunteers"),
    
    F.sum(
        F.when(F.col("has_procedures"), 1).otherwise(0)
    ).alias("facilities_with_procedures"),
    
    F.sum(
        F.when(F.col("has_equipment"), 1).otherwise(0)
    ).alias("facilities_with_equipment"),
    
    F.sum(
        F.when(F.col("has_capability"), 1).otherwise(0)
    ).alias("facilities_with_capability"),
    
    # ========================================
    # SPECIALTY DIVERSITY
    # ========================================
    F.size(
        F.array_distinct(F.flatten(F.collect_list(F.col("specialties"))))
    ).alias("unique_specialties_count"),
    
    # Collect all specialties (will process top 5 later)
    F.flatten(F.collect_list(F.col("specialties"))).alias("all_specialties"),
    
    # ========================================
    # COVERAGE FLAGS - Check if ANY facility has these capabilities
    # ========================================
    F.max(
        F.when(
            F.array_contains(F.col("capability"), "emergency") |
            F.array_contains(F.col("capability"), "emergency care") |
            F.array_contains(F.col("specialties"), "emergencyMedicine"),
            1
        ).otherwise(0)
    ).alias("has_emergency_care"),
    
    F.max(
        F.when(
            F.array_contains(F.col("specialties"), "gynecologyAndObstetrics") |
            F.array_contains(F.col("specialties"), "gynecology") |
            F.array_contains(F.col("capability"), "maternity"),
            1
        ).otherwise(0)
    ).alias("has_maternal_care"),
    
    F.max(
        F.when(
            F.array_contains(F.col("specialties"), "pediatrics") |
            F.array_contains(F.col("capability"), "pediatric"),
            1
        ).otherwise(0)
    ).alias("has_pediatric_care"),
    
    # ========================================
    # OPERATOR TYPE BREAKDOWN
    # ========================================
    F.sum(
        F.when(F.col("operatorTypeId") == "public", 1).otherwise(0)
    ).alias("public_facilities"),
    
    F.sum(
        F.when(F.col("operatorTypeId") == "private", 1).otherwise(0)
    ).alias("private_facilities")
)

print(f"✅ Regional aggregations created")
print(f"Regions aggregated: {df_regional.count()}")

## 4. Process Top Specialties per Region

Extract and rank the top 5 most common specialties in each region.

In [0]:
# Process top specialties - explode, count, rank, and get top 5
from pyspark.sql.functions import struct, col, collect_list

# Create a temporary view for specialty processing
df_regional.createOrReplaceTempView("regional_temp")

# Explode specialties and count frequencies
df_specialty_counts = spark.sql("""
SELECT 
    address_stateOrRegion,
    specialty,
    COUNT(*) as specialty_count
FROM regional_temp
LATERAL VIEW OUTER explode(all_specialties) AS specialty
WHERE specialty IS NOT NULL
GROUP BY address_stateOrRegion, specialty
""")

# Rank specialties within each region and take top 5
window_spec = Window.partitionBy("address_stateOrRegion").orderBy(F.desc("specialty_count"))

df_top_specialties = df_specialty_counts \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") <= 5) \
    .groupBy("address_stateOrRegion") \
    .agg(
        F.collect_list(
            F.struct(
                F.col("specialty").alias("name"),
                F.col("specialty_count").alias("count")
            )
        ).alias("top_specialties")
    )

# Join back to main regional table
df_regional = df_regional.join(
    df_top_specialties,
    on="address_stateOrRegion",
    how="left"
)

print("✅ Top specialties processed")

## 5. Medical Desert Detection

Apply medical desert criteria:
* Flag 1: Total facilities < 3
* Flag 2: Zero hospitals AND < 2 clinics
* Flag 3: Zero doctors AND zero bed capacity

In [0]:
# Add medical desert flagging logic
df_regional = df_regional.withColumn(
    "desert_flag_low_facilities",
    F.when(F.col("total_facilities") < 3, True).otherwise(False)
)

df_regional = df_regional.withColumn(
    "desert_flag_no_hospitals",
    F.when(
        (F.col("total_hospitals") == 0) & (F.col("total_clinics") < 2),
        True
    ).otherwise(False)
)

df_regional = df_regional.withColumn(
    "desert_flag_no_capacity",
    F.when(
        (F.col("total_doctors") == 0) & (F.col("total_bed_capacity") == 0),
        True
    ).otherwise(False)
)

# Overall medical desert flag (ANY condition triggers it)
df_regional = df_regional.withColumn(
    "medical_desert_flag",
    F.col("desert_flag_low_facilities") |
    F.col("desert_flag_no_hospitals") |
    F.col("desert_flag_no_capacity")
)

# Desert severity score (0-3, higher = worse)
df_regional = df_regional.withColumn(
    "desert_severity_score",
    F.col("desert_flag_low_facilities").cast("int") +
    F.col("desert_flag_no_hospitals").cast("int") +
    F.col("desert_flag_no_capacity").cast("int")
)

print("✅ Medical desert flags added")

## 6. Add Metadata and Final Cleanup

In [0]:
# Add metadata
df_regional = df_regional \
    .withColumn("aggregation_timestamp", F.current_timestamp()) \
    .withColumn("source_layer", F.lit("silver")) \
    .drop("all_specialties")  # Drop intermediate column

print("✅ Metadata added and cleanup complete")

## 7. Write to Gold Table

In [0]:
# Write to Delta table
full_table_name = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"

try:
    df_regional.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full_table_name)
    
    print(f"✅ Gold table created: {full_table_name}")
    print(f"   Regions: {df_regional.count()}")
    print(f"   Mode: OVERWRITE")
    
except Exception as e:
    print(f"❌ Error writing Gold table: {e}")
    raise

In [0]:
%sql
-- Add table comment and properties
COMMENT ON TABLE virtue_foundation.ghana.regional_summary IS
'Regional aggregations of healthcare facilities in Ghana.
Includes facility counts, capacity metrics, service availability, and medical desert flags.
Generated from Silver layer with comprehensive coverage analysis.';

ALTER TABLE virtue_foundation.ghana.regional_summary
SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true',
  'layer' = 'gold',
  'quality' = 'aggregated',
  'grain' = 'regional'
);

## 8. Verification and Analysis

In [0]:
# Read back and verify
df_verify = spark.table(full_table_name)

print("="*80)
print("REGIONAL SUMMARY - KEY METRICS")
print("="*80)
print(f"Total regions: {df_verify.count()}")
print(f"Total facilities across Ghana: {df_verify.agg(F.sum('total_facilities')).collect()[0][0]:,}")
print(f"Total hospitals: {df_verify.agg(F.sum('total_hospitals')).collect()[0][0]}")
print(f"Total clinics: {df_verify.agg(F.sum('total_clinics')).collect()[0][0]}")
print(f"Total bed capacity: {df_verify.agg(F.sum('total_bed_capacity')).collect()[0][0]:,}")
print(f"Total doctors: {df_verify.agg(F.sum('total_doctors')).collect()[0][0]:,}")
print("="*80)

In [0]:
# Medical desert analysis
print("\nMEDICAL DESERT ANALYSIS:")
print("="*80)

desert_count = df_verify.filter(F.col("medical_desert_flag") == True).count()
desert_pct = (desert_count / df_verify.count()) * 100

print(f"Medical deserts identified: {desert_count} regions ({desert_pct:.1f}%)")

if desert_count > 0:
    print(f"\nDesert regions by severity:")
    severity_dist = df_verify.filter(F.col("medical_desert_flag") == True) \
        .groupBy("desert_severity_score") \
        .count() \
        .orderBy(F.desc("desert_severity_score"))
    display(severity_dist)
    
    print(f"\nMost critical deserts (severity = 3):")
    critical = df_verify.filter(F.col("desert_severity_score") == 3) \
        .select(
            "address_stateOrRegion",
            "total_facilities",
            "total_hospitals",
            "total_clinics",
            "total_doctors",
            "total_bed_capacity"
        ) \
        .orderBy("total_facilities")
    display(critical)
else:
    print("✅ No medical deserts detected!")

In [0]:
# Regional rankings
print("\nTOP 10 BEST COVERED REGIONS:")
print("="*80)

top_regions = df_verify \
    .filter(F.col("medical_desert_flag") == False) \
    .select(
        "address_stateOrRegion",
        "total_facilities",
        "total_hospitals",
        "total_doctors",
        "total_bed_capacity",
        "unique_specialties_count",
        "avg_completeness_score"
    ) \
    .orderBy(F.desc("total_facilities")) \
    .limit(10)

display(top_regions)

In [0]:
# Service coverage analysis
print("\nSERVICE COVERAGE ACROSS REGIONS:")
print("="*80)

coverage_stats = df_verify.agg(
    F.sum(F.when(F.col("has_emergency_care") == 1, 1).otherwise(0)).alias("regions_with_emergency"),
    F.sum(F.when(F.col("has_maternal_care") == 1, 1).otherwise(0)).alias("regions_with_maternal"),
    F.sum(F.when(F.col("has_pediatric_care") == 1, 1).otherwise(0)).alias("regions_with_pediatric"),
    F.count("*").alias("total_regions")
).collect()[0]

total = coverage_stats['total_regions']
print(f"Emergency care available: {coverage_stats['regions_with_emergency']}/{total} regions ({coverage_stats['regions_with_emergency']/total*100:.1f}%)")
print(f"Maternal care available: {coverage_stats['regions_with_maternal']}/{total} regions ({coverage_stats['regions_with_maternal']/total*100:.1f}%)")
print(f"Pediatric care available: {coverage_stats['regions_with_pediatric']}/{total} regions ({coverage_stats['regions_with_pediatric']/total*100:.1f}%)")

# Regions lacking critical services
print(f"\nREGIONS LACKING CRITICAL SERVICES:")
lacking = df_verify.filter(
    (F.col("has_emergency_care") == 0) |
    (F.col("has_maternal_care") == 0) |
    (F.col("has_pediatric_care") == 0)
).select(
    "address_stateOrRegion",
    "total_facilities",
    F.when(F.col("has_emergency_care") == 0, "❌").otherwise("✅").alias("Emergency"),
    F.when(F.col("has_maternal_care") == 0, "❌").otherwise("✅").alias("Maternal"),
    F.when(F.col("has_pediatric_care") == 0, "❌").otherwise("✅").alias("Pediatric")
).orderBy("total_facilities")

display(lacking)

## ✅ Gold Regional Summary Complete!

**What's been created:**
* ✅ Regional aggregations table: `virtue_foundation.ghana.regional_summary`
* ✅ Facility counts by type (hospitals, clinics, pharmacies, etc.)
* ✅ Capacity metrics (beds, doctors)
* ✅ Service availability flags (emergency, maternal, pediatric care)
* ✅ Top 5 specialties per region
* ✅ Medical desert flags with severity scoring
* ✅ Data quality metrics per region

**Medical Desert Criteria Applied:**
1. Total facilities < 3
2. Zero hospitals AND < 2 clinics
3. Zero doctors AND zero bed capacity

**Key Findings:**
* Check medical desert analysis above
* Review service coverage gaps
* Identify regions needing intervention

**Next Steps:**
1. **Query with Genie:** Ask natural language questions
   * "Which regions are medical deserts?"
   * "Show me regions with no emergency care"
   * "Which areas have the most facilities?"

2. **Create Vector Search index** (`04_Vector_Search_Setup`)

3. **Run Agent pipelines:**
   * IDP Enrichment Agent
   * Medical Desert Detection Agent
   * Anomaly Detection Agent
   * RAG Query Agent

4. **Deploy Frontend:** Build the web dashboard with maps

**Sample Queries:**
```sql
-- All medical deserts
SELECT * FROM virtue_foundation.ghana.regional_summary
WHERE medical_desert_flag = true
ORDER BY desert_severity_score DESC;

-- Regions with critical gaps
SELECT address_stateOrRegion, total_facilities, total_hospitals,
       has_emergency_care, has_maternal_care, has_pediatric_care
FROM virtue_foundation.ghana.regional_summary
WHERE total_hospitals = 0 OR has_emergency_care = 0;

-- Best covered regions
SELECT address_stateOrRegion, total_facilities, total_hospitals,
       total_doctors, total_bed_capacity, unique_specialties_count
FROM virtue_foundation.ghana.regional_summary
WHERE medical_desert_flag = false
ORDER BY total_facilities DESC
LIMIT 10;
```